# MP 9: Gold standard, weak supervision, retrieval-augmented classification and cohort convolution (NARMS E. coli, tetracycline)
Project-track adaptation of MP 9, continuing MP 7 (`mp7_core`, `mp7_narms`). Gold objects are isolates from the
2020+ test period; the cohort axis is collection year; the demonstration / retrieval pool is 2013–2019.

**Order:** run §1 once → annotate by hand (§2, outside this notebook) → run everything from §3 on.
No cell in this notebook assigns a gold label.

In [ ]:
# 0. CPU-only, deterministic setup
import os, warnings
os.environ["CUDA_VISIBLE_DEVICES"] = ""; os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore", message="label space has")      # binary label space: documented deviation
import json, random, numpy as np, pandas as pd
random.seed(0); np.random.seed(0)
import torch; torch.manual_seed(0); torch.use_deterministic_algorithms(True, warn_only=True)
import mp7_core as M, mp7_narms as N, mp9_narms as P
print(json.dumps(M.environment_report(), indent=2))

In [ ]:
# Same raw file, config and split as MP 7
RAW_PATH = "data/CVM-NARMS-Cecal.xlsx"
RAW_SHA = M.sha256_file(RAW_PATH)
raw = N.load_raw(RAW_PATH)
cfg = N.NarmsConfig()
df, drugs, clean_rep = N.prepare(raw, cfg)
TRAIN_MAX, TEST_MIN = 2019, 2020
train_df, test_df = N.temporal_split(df, cfg, TRAIN_MAX, TEST_MIN)
pool = N.to_examples(train_df, cfg, drugs)
N.assert_no_target_leak(pool)
print(raw.shape, "| feature drugs:", drugs, "| pool:", len(pool), "| test-period isolates:", len(test_df))
# Check how the tetracycline MIC is stored (qualifier inside the value, e.g. '<=4', or in its own column?)
print(raw.filter(regex=r"^TET").columns.tolist()); print(df["TET"].astype(str).value_counts().head(10))

## 1. Gold candidates and blank annotation sheets (run ONCE)
50 test-period isolates, proportional by host species, ~30% with an uncensored borderline MIC (4 or 16).
If the MIC qualifier ('<=', '>') is stored in its own column, set `SIGN_COL`.
If the course gave you `overlap_ids.txt`, put it in this folder first; otherwise 20 ids are drawn here.

In [ ]:
SIGN_COL = None          # e.g. "TET Sign" if your file has one
if not os.path.exists("annotation_sheet.csv"):
    gold_ids = P.select_gold_candidates(test_df, cfg, n=50, mic_col="TET", sign_col=SIGN_COL)
    sheet = P.make_annotation_sheet(df, gold_ids, cfg, drugs)
    sheet.to_csv("annotation_sheet.csv", index=False)
    if not os.path.exists("overlap_ids.txt"):
        ov = sorted(np.random.default_rng(P.SEED).choice(gold_ids, 20, replace=False).tolist())
        open("overlap_ids.txt", "w").write("\n".join(ov) + "\n")
    ov = [l.strip() for l in open("overlap_ids.txt") if l.strip()]
    sheet[sheet.item_id.isin(ov)].to_csv("partner_sheet.csv", index=False)
    print("wrote annotation_sheet.csv (50 rows) and partner_sheet.csv (20 rows)")
else:
    print("annotation_sheet.csv already exists: not regenerated")

## 2. Annotate by hand (outside the notebook)
1. Write the start time in `attestation.md`.
2. Copy `annotation_sheet.csv` to `gold_raw.csv` and fill `label`, `borderline`, `quality_flag`, `notes`, `annotator`
   using **only** `codebook.md`. Do not look at `TET SIR`.
3. Write the stop time. Stop at 2 hours even if rows remain (delete unlabeled rows; never add rows).
4. Send `partner_sheet.csv` + `codebook.md` to your partner; save their file as `gold_partner.csv`.

## 3. Gate pre-check and label-reliability audit

In [ ]:
gate = P.precheck_gold_raw("gold_raw.csv")       # also run the course's validate_annotation_submission
print(gate)
mine = P.read_labels_file("gold_raw.csv")
audit = P.audit_vs_lab(mine, df, cfg)            # human label vs. lab interpretation (after annotation)
print("agreement with lab TET SIR:", round(audit.agree.mean(), 3)); display(audit[~audit.agree])

## 4. Inter-annotator agreement (20-object overlap) and adjudication

In [ ]:
partner = P.read_labels_file("gold_partner.csv")
ov = [l.strip() for l in open("overlap_ids.txt") if l.strip()]
iaa = P.kappa_report(mine.set_index("item_id").label.to_dict(), partner.set_index("item_id").label.to_dict(),
                     ov, n_boot=10000)
print({k: v for k, v in iaa.items() if k != "confusion_matrix"}); display(iaa["confusion_matrix"])

In [ ]:
# Fill one decision per disagreement AFTER discussing it with your partner (rule ids from codebook.md §4).
DECISIONS = {
    # "S01234": {"label": "resistant", "rule": "A2", "note": "MIC '>16' is censored above R breakpoint"},
}
adj = P.adjudicate(mine, partner, ov, DECISIONS, out_path="gold_adjudicated.jsonl")
gold = P.gold_examples(df, cfg, drugs, "gold_adjudicated.jsonl")
N.assert_no_target_leak(gold); N.assert_disjoint(pool, gold)
GOLD_SHA = P.sha256_text("\n".join(g.item_id for g in gold))
print(len(gold), adj.adjudication.value_counts().to_dict(), pd.Series([g.label for g in gold]).value_counts().to_dict())

## 5. Weak supervision
Training-period isolates are split once: 20% **dev** (estimates LF weights and host priors, supplies the MP 7
classifier's demonstrations) and the rest **unlabeled** (labels hidden; 2,000 sampled as the silver pool).
The MP 7 LF is the kNN classifier (k = 8 per class) for CPU cost; state this in the report.

In [ ]:
tr = train_df.sort_values(cfg.id_col)
rng = np.random.default_rng(P.SEED)
is_dev = rng.random(len(tr)) < 0.20
dev_df, unl_df = tr[is_dev], tr[~is_dev]
silver_df = unl_df.iloc[np.sort(rng.choice(len(unl_df), min(2000, len(unl_df)), replace=False))]
emb = M.Embedder("sentence-transformers/all-MiniLM-L6-v2", revision=None)   # same model (and pin) as MP 7
dev_ex = N.to_examples(dev_df, cfg, drugs)
mp7_lf = M.KNNClassifier(emb, n_neighbors=5, temperature=0.05).fit(M.sample_demos(dev_ex, 8, seed=0))
recs = N.to_examples(pd.concat([dev_df, silver_df]), cfg, drugs) + gold
mp7_preds = dict(zip([e.item_id for e in recs], mp7_lf.predict([e.text for e in recs])))

f_dev, f_sil = P.lf_frame(dev_df, cfg, drugs), P.lf_frame(silver_df, cfg, drugs)
f_gold = P.lf_frame(df[df[cfg.id_col].astype(str).isin([g.item_id for g in gold])], cfg, drugs)
y_dev = dev_df.set_index(dev_df[cfg.id_col].astype(str)).label
lfs = P.make_default_lfs(f_dev, y_dev, mp7_preds)
L_dev, L_sil, L_gold = P.apply_lfs(f_dev, lfs), P.apply_lfs(f_sil, lfs), P.apply_lfs(f_gold, lfs)
lf_tab = P.lf_summary(L_dev, y_dev); display(lf_tab)
W = P.fit_lf_weights(L_dev, y_dev); print("weights:", {k: round(v, 3) for k, v in W.items()})
v_sil, v_gold = P.weighted_vote(L_sil, W), P.weighted_vote(L_gold, W)
print("silver records:", P.silver_records(v_sil, L_sil, "silver_pool.jsonl"))
gold_s = pd.Series({g.item_id: g.label for g in gold})
silver_vs_gold = P.agreement(gold_s, v_gold.silver_label.reindex(gold_s.index))
y_sil = silver_df.set_index(silver_df[cfg.id_col].astype(str)).label
silver_vs_lab = P.agreement(y_sil, v_sil.silver_label)
for name, a in (("silver vs gold", silver_vs_gold), ("silver vs lab (pool)", silver_vs_lab)):
    print(name, {k: v for k, v in a.items() if k != "confusion_matrix"}); display(a["confusion_matrix"])

In [ ]:
# Failure modes: where does silver go wrong on the pool?
fm = f_sil.assign(lab=y_sil, silver=v_sil.silver_label, wrong=lambda d: d.silver.notna() & (d.silver != d.lab))
display(fm.groupby("host").wrong.mean().round(3).rename("error rate by host"))
display(fm.groupby(["lab", "silver"]).size().rename("n"))
print("abstained:", int(v_sil.silver_label.isna().sum()))

## 6. Retrieval-augmented classification vs. the MP 7 fixed-demonstration baseline
Same LLM, template, verbalizers and constrained scoring as MP 7. **k = total number of demonstrations for both
systems** (the fixed baseline draws k in total, class-balanced as far as possible), so retrieval is the only change.
Variants: plain top-k, and class-balanced top-k. Evaluation set = adjudicated gold.

In [ ]:
llm = M.HFBackend("Qwen/Qwen2.5-0.5B-Instruct", revision=None)      # same model + revision as MP 7
idx = P.ExemplarIndex(emb, pool)                                    # FAISS IndexFlatIP over the 2013-2019 pool
idx_bal = P.ExemplarIndex(emb, pool, by_label=True)
KS, SEEDS = (3, 5, 10), (0, 1, 2)
rac_runs = P.run_rac_sweep(llm, idx, idx_bal, pool, gold, ks=KS, seeds=SEEDS,
                           out_path="predictions_mp9.jsonl", eval_sha256=GOLD_SHA)
json.dump(rac_runs, open("rac_runs.json", "w"), indent=1, default=str)

In [ ]:
# Every number below is re-derived from predictions_mp9.jsonl
rac_tab = P.metric_table("predictions_mp9.jsonl", n_boot=2000)
display(rac_tab.pivot_table(index=["classifier", "k"], columns="metric", values=["mean", "ci95_lo", "ci95_hi"]).round(3))
diffs = [P.paired_bootstrap_diff("predictions_mp9.jsonl", a, "fixed_llm", k, metric=m)
         | {"metric": mn}
         for a in ("retrieval_llm", "retrieval_llm_balanced") for k in KS
         for mn, m in (("macro_f1", M.dev_macro_f1), ("resistant_recall", P.resistant_recall))]
diff_tab = pd.DataFrame(diffs); display(diff_tab.round(3))
display(pd.DataFrame(rac_runs).groupby(["classifier", "k"])[["max_pred_share", "demo_label_share_resistant"]].mean().round(3))

In [ ]:
# Retrieval quality: P@k / Recall@k against the codebook relevance rule (and human judgments if present)
ranked = P.ranked_neighbors(idx, gold, kmax=10)
f_pool = P.lf_frame(train_df, cfg, drugs)
pool_lab = {e.item_id: e.label for e in pool}; gold_lab = {g.item_id: g.label for g in gold}
is_rel, n_rel_total = P.rule_relevance(f_gold, f_pool, gold_lab, pool_lab, min_jaccard=0.5)
rel_rule = {q: {c: int(is_rel(q, c)) for c in cs} for q, cs in ranked.items()}
tot = {q: n_rel_total(q) for q in ranked}
ret_tab = P.retrieval_metrics(ranked, rel_rule, ks=KS, n_relevant_total=tot); display(ret_tab.round(3))
if not os.path.exists("relevance_sheet.csv"):          # optional human judgments (does not count as gold rows)
    P.relevance_sheet(ranked, f_pool, {e.item_id: e.text for e in pool}, {g.item_id: g.text for g in gold}
                      ).to_csv("relevance_sheet.csv", index=False)
if os.path.exists("relevance_judged.csv"):
    rel_h = P.read_relevance("relevance_judged.csv")
    ranked_h = {q: ranked[q] for q in rel_h}
    ret_tab_h = P.retrieval_metrics(ranked_h, rel_h, ks=KS); display(ret_tab_h.round(3))
    pairs = [(rel_h[q][c], rel_rule[q][c]) for q in rel_h for c in rel_h[q]]
    from sklearn.metrics import cohen_kappa_score
    print("human vs rule relevance kappa:", cohen_kappa_score(*zip(*pairs)))

## 7. Cohort convolution along collection year
F = years × drugs resistance rates (tetracycline included: this part is descriptive, not a model input).
Kernels are applied with the cross-correlation convention, so `[-1, 1]` gives F[t+1] − F[t]
(`np.convolve` would flip the sign). Drift magnitude = mean |change| across drugs (pp), with basic-bootstrap CIs
(magnitudes are biased upward by noise); per-drug signed changes use percentile CIs.

In [ ]:
DRIFT_DRUGS = ["TET SIR"] + drugs
drift = P.drift_analysis(df, cfg, DRIFT_DRUGS, n_boot=1000, policy_transition=(2016, 2017))
print("dropped (not tested every year):", drift["dropped_drugs_not_tested_every_year"])
print("isolates per year:", drift["n_per_year"])
display(drift["magnitude"].round(2))
display(drift["per_drug"].query("drug == 'TET SIR'").round(2))
print(drift.get("policy_excess_drift_pp"))
# Optional: the same analysis per host species
by_host = {h: P.drift_analysis(g, cfg, DRIFT_DRUGS, n_boot=500)["per_drug"].query("drug == 'TET SIR'")
           for h, g in df.groupby(cfg.source_col)}

In [ ]:
import matplotlib.pyplot as plt
r = drift["rates"].query("drug == 'TET SIR'")
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.fill_between(r.year, 100 * r.ci95_lo, 100 * r.ci95_hi, alpha=.25, label="95% CI")
ax.plot(r.year, 100 * r.rate, marker="o", label="yearly rate")
ax.plot(drift["years"][1:-1], 100 * drift["S"][:, drift["drugs"].index("TET SIR")], label="smoothed [0.25, 0.5, 0.25]")
ax.axvline(2016.5, ls="--", c="gray"); ax.text(2016.6, ax.get_ylim()[1] * .95, "GFI #213", fontsize=8)
ax.set_xlabel("collection year"); ax.set_ylabel("tetracycline resistant (%)"); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig("fig_tet_drift.png", dpi=150)

## 8. CIS 731: learned 1-D convolutional kernel vs. fixed kernels
One `Conv1d(1, 1, kernel_size=3)` shared across drugs, trained to forecast next year's rate from the previous
three (ridge pull towards persistence). Rolling origin over the last 2 years: the kernel for test year t is fit
on years < t only. The closed-form fit is a check on the PyTorch fit.

In [ ]:
fc = P.forecast_comparison(drift["F"], drift["years"], w=3, n_test_years=2, use_torch=True)
display(fc["table"].round(3)); print(fc["kernels"])
fc_check = P.forecast_comparison(drift["F"], drift["years"], w=3, n_test_years=2, use_torch=False)
print("closed-form check:", fc_check["kernels"])

## 9. results.json and environment record

In [ ]:
results = {
    "meta": {"raw_sha256": RAW_SHA, "gold_ids_sha256": GOLD_SHA, "split": [TRAIN_MAX, TEST_MIN],
             "llm": [llm.model_id, llm.revision], "embedder": [emb.model_id, emb.revision],
             "environment": M.environment_report(), "cleaning": clean_rep},
    "gold": {"gate": gate, "n_adjudicated": len(gold), "lab_agreement": float(audit.agree.mean())},
    "iaa": iaa,
    "weak_supervision": {"lf_summary_dev": lf_tab, "weights": W, "silver_vs_gold": silver_vs_gold,
                         "silver_vs_lab_pool": silver_vs_lab, "n_silver": len(v_sil)},
    "rac": {"metrics": rac_tab, "paired_diffs_vs_fixed": diff_tab, "retrieval_rule": ret_tab},
    "convolution": {k: drift[k] for k in ("years", "drugs", "magnitude", "per_drug", "n_per_year")}
                   | {"policy_excess_drift_pp": drift.get("policy_excess_drift_pp")},
    "cis731_forecast": {"table": fc["table"], "kernels": fc["kernels"], "test_years": fc["test_years"]},
}
P.write_results("results.json", results); print("wrote results.json")